# Team A — Expanded-data model training (new experiment)

This unexecuted copy is separate from the submitted notebook/report. Original
datasets and saved models remain unchanged. It uses 4,989,171 NEW synthetic
transactions from 2022–2026, for 999 new cards, across all 12 Sparkov profiles.
The original 2019–2020 transactions are NOT included.

Year files were explored/scored before training. They are development data,
not an untouched final test. A new future test set for these same customers is
still pending. This notebook reports chronological validation results only.
Training is separate from the demo and saves new model files.


## 1. Locate the current project code

Run locally from the Phase III folder, or in Colab clone the current integration
branch. The old executed notebook embeds historical v2 code; this copy imports
the current v3 modules instead. Colab needs the combined CSV copied/uploaded to
its runtime or mounted Drive. Repository code does not include the CSVs.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
IN_COLAB = Path('/content').exists()
if IN_COLAB:
    checkout = Path('/content/CAI2820C_FraudShield')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--branch', 'phase-iii-history-context',
                        '--single-branch', 'https://github.com/MCDR-Code/CAI2820C_FraudShield.git',
                        str(checkout)], check=True)
    PROJECT = checkout / 'Part_III_Modeling_Demo'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
else:
    PROJECT = Path.cwd()
    if not (PROJECT / 'train.py').exists():
        PROJECT = PROJECT / 'Part_III_Modeling_Demo'
assert (PROJECT / 'train.py').exists(), 'Set PROJECT to the Phase III folder.'
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
from train import train
from core import load_bundle
from history import HISTORY_VERSION
from shared.project_data import sha256
import pandas as pd
print('Project:', PROJECT)
print('History version:', HISTORY_VERSION)


## 2. Data location and experiment settings

The combined training file is `data/expanded_training_2022_2026/fraudTrain.csv`.
On this Windows computer it is already prepared. For Colab, mount Drive and
change DATA_DIR below to the folder containing that file. Optional Drive mounting:
`from google.colab import drive; drive.mount('/content/drive')`.

All three algorithms are selectable here. The transaction-model experiment fits
Logistic Regression, Random Forest and Gradient Boosting on identical fitting/
validation rows. The second experiment fits Gradient Boosting with v3 history.
Remove an experiment/model from this dictionary if you do not want to fit it.
Use a new RUN_NAME for each run; existing model folders are never overwritten.


In [ ]:
DATA_DIR = PROJECT / 'data/expanded_training_2022_2026'
RUN_NAME = 'expanded_2022_2026_run1'
ARTIFACTS = PROJECT / 'artifacts' / RUN_NAME
EXPERIMENTS = {
    'transaction_models': (False, ('Logistic Regression', 'Random Forest', 'Gradient Boosting')),
    'history_v3': (True, ('Gradient Boosting',)),
}
assert (DATA_DIR / 'fraudTrain.csv').exists(), 'Set DATA_DIR or copy/upload fraudTrain.csv first.'
manifest_path = DATA_DIR / 'training_data_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert sha256(DATA_DIR / 'fraudTrain.csv') == manifest['sha256'], 'Training CSV differs from saved manifest.'
    display(pd.Series({key: manifest[key] for key in ['rows', 'cards', 'fraud', 'original_data_included']}))
print('Future test present:', (DATA_DIR / 'fraudTest.csv').exists())


## 3. Train once and save new models

The earliest approximately 80% of transactions are fitting data; the latest
approximately 20% are validation. Transactions sharing a timestamp remain on the
same side of the boundary. History uses only strictly earlier transactions,
including earlier fitting-period context during validation. No target labels
are used as features. Thresholds/model selection use validation only.

The explicit expanded-data setting permits this new dataset size and a pending
future test file. Original training defaults retain their original size checks.
If a new fraudTest.csv is supplied, its labels remain unread during training and
its chronology/transaction-ID separation are checked. Do not put the old test
file in this folder. No full-data training has been executed in this notebook yet.


In [ ]:
experiments = {}
for name, (use_history, models) in EXPERIMENTS.items():
    output = ARTIFACTS / name
    locked, comparison = train(DATA_DIR, output, history=use_history,
                               candidates_to_fit=models, expanded_data=True)
    experiments[name] = {'metadata': locked, 'comparison': comparison}
    display(comparison)


## 4. Review validation results and saved-model checks

Precision describes correct alerts; recall describes detected fraud. Inspect
false alerts and missed fraud alongside both measures. Compare these new
experiments on the SAME validation rows. The old Random Forest's published
metrics used different data and cannot directly measure this experiment's gain.
Each experiment saves its selected model, threshold, data fingerprints,
validation comparison and model-sharing manifest. It does not save every fitted
candidate as a separate demo selector entry.


In [ ]:
from model_sharing import check_model
for name in experiments:
    print(name)
    display(pd.read_csv(ARTIFACTS / name / 'validation_comparison.csv'))
    print(check_model(ARTIFACTS / name))


## 5. History error analysis and inference parity

These checks use this experiment's validation period, not a future test set.
They run only if the history experiment above was selected.


In [ ]:
if 'history_v3' in experiments:
    from verify_context import verify
    from analyze_errors import main as analyze_errors
    display(pd.Series(verify(DATA_DIR, ARTIFACTS / 'history_v3')))
    analyze_errors(ARTIFACTS / 'history_v3', DATA_DIR)


## 6. Future final test and demo integration

Next: generate new transactions after December 31, 2026 using the SAME saved
customer records/profiles; save them separately as fraudTest.csv. Freeze models
and thresholds before evaluating its fraud labels. Do not reuse 2022–2026 rows
as final test data after training on them.

Models saved by this notebook are separate new experiment artifacts. They are
not automatically installed over the demo baseline or history model. Preserve
the original Random Forest for comparison and explicitly choose the new model
after examining the run. Transfer trusted saved models with matching code and
package versions; follow Training_and_Model_Sharing.md.
